# Módulo 3 — Camera-Ready WITCOM 2026 (rol CE)
Notebook de trabajo: baselines, regeneración class-blind (3 técnicas), recálculo de Tabla 11 y (próximamente) CNN.
Cada sección corresponde a una tarea de la bitácora `Bitacora_Modulo3_CE.md`.

## Etapa 0 — Setup

In [4]:
import os, joblib, numpy as np
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

# --- Rutas centrales del proyecto (ajusta DRIVE_ROOT si tu carpeta 'TT' esta en otro lugar) ---
INPUT_ROOT = '/kaggle/input/datasets/palaciosroblesdiego/data-files/TT'
UMAP_PKL_PATH = os.path.join(INPUT_ROOT, 'data/shared/umap_params.pkl')
BETO_PKL_PATH = os.path.join(INPUT_ROOT, 'data/shared/Exp_4_1_BETO_HPO_embeddings.pkl')
ORIGINAL_IMAGES_BASE = os.path.join(DRIVE_ROOT, 'images')                       # imagenes coloreadas originales

CLASSBLIND_BASE = '/kaggle/working/images_grayscale_classblind'       # imagenes nuevas class-blind
NESTED_BASE = os.path.join(CLASSBLIND_BASE, 'gray_scott_nested_cv')
os.makedirs(NESTED_BASE, exist_ok=True)

umap_data = joblib.load(UMAP_PKL_PATH)
for split in ['train', 'val', 'test']:
    p = umap_data[split]['params']
    print(f'{split}: shape={p.shape}, min={p.min():.4f}, max={p.max():.4f}')
    assert p.min() >= 0.0 and p.max() <= 1.0, f'{split}: parametros fuera de [0,1]'


train: shape=(908, 5), min=0.0000, max=1.0000
val: shape=(114, 5), min=0.0000, max=1.0000
test: shape=(114, 5), min=0.0000, max=1.0000


## Etapa 1 — Baselines directos sobre 5D UMAP (control de circularidad)
Tarea 1.2: clasificadores directos sobre los 5 componentes UMAP, sin pasar por ninguna imagen.

In [5]:
# ============================================================
# Tarea 1.2 - Baselines directos sobre 5D UMAP
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, classification_report

RANDOM_STATE = 42

X_train, y_train = umap_data['train']['params'], umap_data['train']['labels']
X_val,   y_val   = umap_data['val']['params'],   umap_data['val']['labels']
X_test,  y_test  = umap_data['test']['params'],  umap_data['test']['labels']

classifiers = {
    'LogReg': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'),
    'SVM':    SVC(kernel='rbf', random_state=RANDOM_STATE, class_weight='balanced'),
    'RF':     RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight='balanced'),
    'MLP':    MLPClassifier(hidden_layer_sizes=(32,), max_iter=2000, random_state=RANDOM_STATE),
}

results_5d = {}
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    f1_val  = f1_score(y_val,  clf.predict(X_val),  average='macro')
    f1_test = f1_score(y_test, clf.predict(X_test), average='macro')
    results_5d[name] = {'F1_val': f1_val, 'F1_test': f1_test}

print(f"{'Modelo':<10} {'F1-Val':<10} {'F1-Test':<10}")
for name, r in results_5d.items():
    print(f"{name:<10} {r['F1_val']:<10.4f} {r['F1_test']:<10.4f}")


Modelo     F1-Val     F1-Test   
LogReg     0.7004     0.7246    
SVM        0.6902     0.7246    
RF         0.6874     0.7304    
MLP        0.6826     0.6933    


## Etapa 2.0 — Diagnóstico del dataset de imágenes ORIGINAL (coloreado)
Referencia / verificación de integridad. No es necesario para el CNN, pero documenta la fuga de color detectada.

In [6]:
# Verificacion de integridad del dataset original (coloreado) en Drive
for tecnica in ['gray_scott', 'lorenz', 'perlin']:
    print(f'\n=== {tecnica} ===')
    for split in ['train', 'val', 'test']:
        for clase in ['depresivo', 'no_depresivo']:
            path = os.path.join(ORIGINAL_IMAGES_BASE, tecnica, split, clase)
            if os.path.exists(path):
                n = len([f for f in os.listdir(path) if f.endswith('.png')])
                print(f'  {split}/{clase}: {n} imagenes')
            else:
                print(f'  {split}/{clase}: NO EXISTE ({path})')



=== gray_scott ===
  train/depresivo: 353 imagenes
  train/no_depresivo: 555 imagenes
  val/depresivo: 44 imagenes
  val/no_depresivo: 70 imagenes
  test/depresivo: 44 imagenes
  test/no_depresivo: 70 imagenes

=== lorenz ===
  train/depresivo: 353 imagenes
  train/no_depresivo: 555 imagenes
  val/depresivo: 44 imagenes
  val/no_depresivo: 70 imagenes
  test/depresivo: 44 imagenes
  test/no_depresivo: 70 imagenes

=== perlin ===
  train/depresivo: 353 imagenes
  train/no_depresivo: 555 imagenes
  val/depresivo: 44 imagenes
  val/no_depresivo: 70 imagenes
  test/depresivo: 44 imagenes
  test/no_depresivo: 70 imagenes


In [7]:
# ============================================================
# Tarea 2.1 - Test de fuga por color/luminancia (Gray-Scott, dataset completo)
# ============================================================
from PIL import Image
from scipy import stats

BASE_PATH_LEAK_TEST = os.path.join(ORIGINAL_IMAGES_BASE, 'gray_scott', 'train')

def collect_gray_stats(class_folder):
    means, stds = [], []
    folder = os.path.join(BASE_PATH_LEAK_TEST, class_folder)
    for fname in os.listdir(folder):
        if not fname.endswith('.png'):
            continue
        img = Image.open(os.path.join(folder, fname)).convert('L')
        arr = np.array(img).astype(float)
        means.append(arr.mean()); stds.append(arr.std())
    return np.array(means), np.array(stds)

mean_dep, std_dep = collect_gray_stats('depresivo')
mean_nodep, std_nodep = collect_gray_stats('no_depresivo')

print(f'N depresivo: {len(mean_dep)}, N no_depresivo: {len(mean_nodep)}')
print(f'Media gris  -> dep: {mean_dep.mean():.2f}+-{mean_dep.std():.2f} | nodep: {mean_nodep.mean():.2f}+-{mean_nodep.std():.2f}')
t_mean, p_mean = stats.ttest_ind(mean_dep, mean_nodep, equal_var=False)
print(f'Welch t-test (media gris): t={t_mean:.2f}, p={p_mean:.4f}')

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
X_leak = np.concatenate([mean_dep, mean_nodep]).reshape(-1, 1)
y_leak = np.concatenate([np.ones(len(mean_dep)), np.zeros(len(mean_nodep))])
clf_leak = LogisticRegression().fit(X_leak, y_leak)
print(f'F1-Macro clasificador trivial (solo brillo): {f1_score(y_leak, clf_leak.predict(X_leak), average="macro"):.4f}')
# Resultado ya documentado en bitacora: F1=0.5077 -> decision: regenerar class-blind de todas formas


N depresivo: 353, N no_depresivo: 555
Media gris  -> dep: 227.02+-9.67 | nodep: 221.32+-12.95
Welch t-test (media gris): t=7.56, p=0.0000
F1-Macro clasificador trivial (solo brillo): 0.5077


## Etapa 2.2 — Regeneración class-blind (3 técnicas)
### 2.2.a Gray-Scott

In [8]:
# ============================================================
# Gray-Scott class-blind -- CODIGO ORIGINAL de Marco (data/shared),
# con UN SOLO cambio: render final directo a gris, sin colormap por clase.
# ============================================================
def gray_scott_step(U, V, Du, Dv, f, k, dt=1.0):
    lap_U = (np.roll(U,1,0)+np.roll(U,-1,0)+np.roll(U,1,1)+np.roll(U,-1,1)-4*U)
    lap_V = (np.roll(V,1,0)+np.roll(V,-1,0)+np.roll(V,1,1)+np.roll(V,-1,1)-4*V)
    uvv = U * V * V
    return U + dt*(Du*lap_U - uvv + f*(1-U)), V + dt*(Dv*lap_V + uvv - (f+k)*V)

def run_gray_scott(f, k, size=256, n_iter=3000, Du=0.16, Dv=0.08, seed_radius=5, dt=1.0):
    U = np.ones((size, size)); V = np.zeros((size, size))
    cx, cy = size // 2, size // 2
    r = seed_radius
    U[cx-r:cx+r, cy-r:cy+r] = 0.50
    V[cx-r:cx+r, cy-r:cy+r] = 0.25
    rng = np.random.default_rng(42)  # semilla FIJA (igual para todas las muestras) -- no depende de label ni indice
    U += rng.uniform(-0.01, 0.01, U.shape)
    V += rng.uniform(-0.01, 0.01, V.shape)
    U, V = np.clip(U, 0, 1), np.clip(V, 0, 1)
    for _ in range(n_iter):
        U, V = gray_scott_step(U, V, Du, Dv, f, k, dt)
    return V

def umap_to_rd_params(umap_vec):
    u0, u1, u2, u3, u4 = umap_vec
    f = 0.022 + u0 * (0.046 - 0.022)
    k = 0.059 + u1 * (0.066 - 0.059)
    Dv = 0.04 + u2 * (0.08 - 0.04)
    seed_radius = int(3 + u3 * (18 - 3))
    n_iter = int(2000 + u4 * (7000 - 2000))  # NOTA: codigo real usa tope 7000 (el paper dice 9000 -- reconciliar en texto)
    return dict(f=f, k=k, Du=0.16, Dv=Dv, seed_radius=seed_radius, n_iter=n_iter)

def run_gray_scott_safe(f, k, Du, Dv, seed_radius, n_iter, size=256, max_retries=3):
    for intento in range(max_retries):
        V = run_gray_scott(f=f, k=k, Du=Du, Dv=Dv, seed_radius=seed_radius, n_iter=n_iter, size=size)
        if V.var() > 1e-4:
            return V, intento
        Dv = max(0.04, Dv * 0.85)
        seed_radius = min(25, seed_radius + 4)
    V = run_gray_scott(f=0.035, k=0.061, Du=0.16, Dv=0.06, seed_radius=10, n_iter=5000, size=size)
    return V, max_retries

def generate_and_save_grayscale(idx, umap_vec, label, split_name, output_dir):
    """VERSION CLASS-BLIND: renderiza V directo a gris, sin colormap por clase."""
    params = umap_to_rd_params(umap_vec)                      # <-- SOLO umap_vec, nunca el label
    V, reintentos = run_gray_scott_safe(**params, size=256)    # <-- SOLO parametros fisicos
    V_norm = (V - V.min()) / (V.max() - V.min() + 1e-8)
    gray = (V_norm * 255).astype(np.uint8)
    fname = f'rd_{split_name}_{idx:04d}_label{label}.png'
    fpath = os.path.join(output_dir, fname)
    Image.fromarray(gray, mode='L').save(fpath)
    return fname, reintentos, params


In [9]:
# ============================================================
# Tarea 2.2.5 (Gray-Scott) - Generacion completa class-blind (codigo real de Marco + fix gris)
# ============================================================
import time

GS_OUTPUT = os.path.join(CLASSBLIND_BASE, 'gray_scott')
os.makedirs(GS_OUTPUT, exist_ok=True)

total_generated, total_skipped, total_retries = 0, 0, 0
t_start = time.time()
n_total_all_splits = sum(len(umap_data[s]['labels']) for s in ['train','val','test'])

for split in ['train', 'val', 'test']:
    X, y = umap_data[split]['params'], umap_data[split]['labels']
    n = len(y)
    print(f'\n=== Split: {split} ({n} muestras) ===')
    for i in range(n):
        label = int(y[i])
        fname = f'rd_{split}_{i:04d}_label{label}.png'
        fpath = os.path.join(GS_OUTPUT, fname)
        if os.path.exists(fpath):
            total_skipped += 1
            continue
        _, reintentos, _ = generate_and_save_grayscale(i, X[i], label, split, GS_OUTPUT)
        total_generated += 1
        total_retries += reintentos
        if total_generated % 20 == 0:
            elapsed = time.time() - t_start
            rate = elapsed / total_generated
            remaining = (n_total_all_splits - total_generated - total_skipped) * rate
            print(f'  [{split}] {i+1}/{n} generadas={total_generated} (~{rate:.1f}s/img, ETA: {remaining/60:.1f} min)')

print(f'\nGray-Scott listo. Generadas: {total_generated}, saltadas: {total_skipped}, reintentos totales: {total_retries}')



=== Split: train (908 muestras) ===

=== Split: val (114 muestras) ===

=== Split: test (114 muestras) ===

Gray-Scott listo. Generadas: 0, saltadas: 1136, reintentos totales: 0


### 2.2.b Perlin

In [10]:
!pip install noise -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.0/132.0 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [11]:
# ============================================================
# Perlin Flow Fields class-blind -- funciones
# ============================================================
import hashlib
from noise import pnoise2

PERLIN_RANGES = {'frequency':(2.0,10.0), 'turbulence':(0.5,3.0), 'speed':(0.5,3.0),
                 'traj_length':(30,200), 'n_octaves':(1,8)}

def lerp(t, lo, hi):
    return lo + t * (hi - lo)

def map_umap_to_perlin(params):
    return {
        'frequency':   lerp(params[0], *PERLIN_RANGES['frequency']),
        'turbulence':  lerp(params[1], *PERLIN_RANGES['turbulence']),
        'speed':       lerp(params[2], *PERLIN_RANGES['speed']),
        'traj_length': int(lerp(params[3], *PERLIN_RANGES['traj_length'])),
        'n_octaves':   int(lerp(params[4], *PERLIN_RANGES['n_octaves'])),
    }

def seed_from_umap(params):
    key = ','.join(f'{v:.8f}' for v in params)
    return int(hashlib.md5(key.encode()).hexdigest(), 16) % (2**32)

def generate_noise_grid(size, frequency, n_octaves, seed=0):
    scale = frequency / size
    def _noise_at(x, y):
        return pnoise2(x*scale, y*scale, octaves=n_octaves, persistence=0.5, lacunarity=2.0, base=seed)
    xs, ys = np.arange(size), np.arange(size)
    xx, yy = np.meshgrid(xs, ys)
    return np.vectorize(_noise_at)(xx, yy)

def generate_flow_field(noise_grid, turbulence):
    return noise_grid * np.pi * turbulence

def trace_particles(flow_field, n_particles, speed, traj_length, rng):
    h, w = flow_field.shape
    trajectories = []
    for _ in range(n_particles):
        x, y = rng.uniform(0, w-1), rng.uniform(0, h-1)
        path = [(x, y)]
        for _ in range(traj_length):
            xi, yi = int(x) % w, int(y) % h
            angle = flow_field[yi, xi]
            x += np.cos(angle) * speed; y += np.sin(angle) * speed
            if x < 0 or x >= w or y < 0 or y >= h:
                break
            path.append((x, y))
        if len(path) > 1:
            trajectories.append(path)
    return trajectories

def draw_trajectories_grayscale(trajectories, noise_grid, size):
    nmin, nmax = noise_grid.min(), noise_grid.max()
    noise_norm = (noise_grid - nmin) / (nmax - nmin + 1e-8)
    canvas = np.zeros((size, size), dtype=np.float64)
    for path in trajectories:
        sx, sy = int(path[0][0]) % size, int(path[0][1]) % size
        gray_val = noise_norm[sy, sx]
        for i in range(len(path) - 1):
            x0, y0 = int(path[i][0]), int(path[i][1])
            if 0 <= x0 < size and 0 <= y0 < size:
                canvas[y0, x0] = max(canvas[y0, x0], gray_val)
    return (canvas * 255).astype(np.uint8)

def generate_perlin_image_classblind(params, size=256, n_particles=800):
    seed = seed_from_umap(params)
    rng = np.random.default_rng(seed)
    p = map_umap_to_perlin(params)
    noise_grid = generate_noise_grid(size, p['frequency'], p['n_octaves'], seed=seed % 1024)
    flow_field = generate_flow_field(noise_grid, p['turbulence'])
    trajectories = trace_particles(flow_field, n_particles, p['speed'], p['traj_length'], rng)
    return draw_trajectories_grayscale(trajectories, noise_grid, size)


In [12]:
# Generacion completa Perlin class-blind
PERLIN_OUTPUT = os.path.join(CLASSBLIND_BASE, 'perlin')
os.makedirs(PERLIN_OUTPUT, exist_ok=True)

for split in ['train', 'val', 'test']:
    X, y = umap_data[split]['params'], umap_data[split]['labels']
    for i in range(len(y)):
        label = int(y[i])
        fname = f'perlin_{split}_{i:04d}_label{label}.png'
        fpath = os.path.join(PERLIN_OUTPUT, fname)
        if os.path.exists(fpath):
            continue
        img = generate_perlin_image_classblind(X[i])   # <-- SOLO vector UMAP
        Image.fromarray(img, mode='L').save(fpath)
    print(f'{split}: listo')
print('Perlin class-blind generado completo.')


train: listo
val: listo
test: listo
Perlin class-blind generado completo.


### 2.2.c Lorenz

In [13]:
# ============================================================
# Lorenz class-blind -- funciones (de lorenz.py, sin cambios)
# ============================================================
from scipy.integrate import solve_ivp

def lorenz_system(t, state, sigma, rho, beta):
    x, y, z = state
    return [sigma*(y-x), x*(rho-z)-y, x*y-beta*z]

LORENZ_RANGES = {'sigma':(8.0,14.0), 'rho':(24.0,32.0), 'beta':(2.0,4.0),
                 'n_puntos':(5_000,100_000), 'elevacion':(0.0,180.0)}

def map_umap_to_lorenz(umap_components):
    c = np.asarray(umap_components, dtype=np.float64)
    assert c.shape == (5,)
    def _lerp(v, lo, hi): return lo + v * (hi - lo)
    return {
        'sigma':     _lerp(c[0], *LORENZ_RANGES['sigma']),
        'rho':       _lerp(c[1], *LORENZ_RANGES['rho']),
        'beta':      _lerp(c[2], *LORENZ_RANGES['beta']),
        'n_puntos':  int(_lerp(c[3], *LORENZ_RANGES['n_puntos'])),
        'elevacion': _lerp(c[4], *LORENZ_RANGES['elevacion']),
    }

def solve_attractor(sigma, rho, beta, n_puntos, t_max=50.0, initial_state=None, t_discard=5.0):
    if initial_state is None:
        initial_state = [1.0, 1.0, 1.0]
    n_total = int(n_puntos * (t_max / (t_max - t_discard)))
    t_eval = np.linspace(0, t_max, n_total)
    sol = solve_ivp(lorenz_system, (0, t_max), initial_state, args=(sigma, rho, beta),
                     method='RK45', t_eval=t_eval, rtol=1e-9, atol=1e-9)
    if not sol.success:
        raise RuntimeError(f'solve_ivp fallo: {sol.message}')
    mask = sol.t >= t_discard
    return {'x': sol.y[0, mask], 'y': sol.y[1, mask], 'z': sol.y[2, mask], 't': sol.t[mask]}


In [14]:
# ============================================================
# Lorenz class-blind -- renderizado (de render.py, PATCHED: gris real, sin label)
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection

LORENZ_IMG_SIZE = 256
LW_RANGE, ALPHA_RANGE = (0.1, 1.5), (0.3, 0.7)

def compute_velocity(trajectory):
    dx, dy, dz = np.diff(trajectory['x']), np.diff(trajectory['y']), np.diff(trajectory['z'])
    dt = np.diff(trajectory['t'])
    dt = np.where(dt == 0, 1e-12, dt)
    return np.sqrt(dx**2 + dy**2 + dz**2) / dt

def normalize_array(arr):
    vmin, vmax = arr.min(), arr.max()
    if vmax - vmin < 1e-12:
        return np.full_like(arr, 0.5)
    return (arr - vmin) / (vmax - vmin)

def render_attractor_enriched(trajectory, elevacion, output_path=None, azimuth=45.0,
                              lw_range=LW_RANGE, alpha_range=ALPHA_RANGE,
                              invert_alpha=False, bg_color='black'):
    speed = compute_velocity(trajectory)
    speed_norm = normalize_array(speed)
    z = trajectory['z']
    z_mid = (z[:-1] + z[1:]) / 2.0
    z_norm = normalize_array(z_mid)

    gray_val = 0.3 + 0.7 * z_norm   # intensidad de gris (reemplaza el colormap por clase)
    colors_rgba = np.zeros((len(gray_val), 4))
    colors_rgba[:, 0] = colors_rgba[:, 1] = colors_rgba[:, 2] = gray_val

    alpha_min, alpha_max = alpha_range
    alphas = (alpha_max - speed_norm*(alpha_max-alpha_min)) if invert_alpha else (alpha_min + speed_norm*(alpha_max-alpha_min))
    colors_rgba[:, 3] = alphas

    lw_min, lw_max = lw_range
    linewidths = lw_min + speed_norm * (lw_max - lw_min)

    x, y = trajectory['x'], trajectory['y']
    points = np.column_stack([x, y, z]).reshape(-1, 1, 3)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    fig = plt.figure(figsize=(LORENZ_IMG_SIZE/100, LORENZ_IMG_SIZE/100), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1], projection='3d')
    fig.patch.set_facecolor(bg_color); ax.set_facecolor(bg_color)
    lc = Line3DCollection(segments, colors=colors_rgba, linewidths=linewidths)
    ax.add_collection3d(lc)
    ax.set_xlim(x.min(), x.max()); ax.set_ylim(y.min(), y.max()); ax.set_zlim(z.min(), z.max())
    ax.view_init(elev=elevacion, azim=azimuth)
    ax.set_axis_off(); ax.grid(False)
    for pane in (ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane):
        pane.fill = False; pane.set_edgecolor('none')

    if output_path is not None:
        d = os.path.dirname(output_path)
        if d:
            os.makedirs(d, exist_ok=True)
        fig.savefig(output_path, dpi=100, facecolor=bg_color, bbox_inches='tight', pad_inches=0)
        img = Image.open(output_path).convert('L')
        if img.size != (LORENZ_IMG_SIZE, LORENZ_IMG_SIZE):
            img = img.resize((LORENZ_IMG_SIZE, LORENZ_IMG_SIZE))
        img.save(output_path)
    plt.close(fig)
    return fig


In [15]:
# Generacion completa Lorenz class-blind (reanudable)
LORENZ_OUTPUT = os.path.join(CLASSBLIND_BASE, 'lorenz')
os.makedirs(LORENZ_OUTPUT, exist_ok=True)

for split in ['train', 'val', 'test']:
    X, y = umap_data[split]['params'], umap_data[split]['labels']
    n = len(y)
    t0 = time.time()
    for i in range(n):
        label = int(y[i])
        fname = f'lorenz_{split}_{i:04d}_label{label}.png'
        fpath = os.path.join(LORENZ_OUTPUT, fname)
        if os.path.exists(fpath):
            continue
        params = map_umap_to_lorenz(X[i])               # <-- SOLO vector UMAP
        trajectory = solve_attractor(sigma=params['sigma'], rho=params['rho'],
                                       beta=params['beta'], n_puntos=params['n_puntos'])
        render_attractor_enriched(trajectory, elevacion=params['elevacion'], output_path=fpath)
        if (i+1) % 20 == 0:
            print(f'  [{split}] {i+1}/{n} ({time.time()-t0:.0f}s transcurridos)')
    print(f'{split}: listo')
print('Lorenz class-blind generado completo.')


train: listo
val: listo
test: listo
Lorenz class-blind generado completo.


## Etapa 2.2.6 — Recalcular Tabla 11 sobre los datasets class-blind
Entropía, contraste, densidad de bordes y energía LBP por clase y técnica, con prueba de Welch y corrección de Holm.

In [16]:
# ============================================================
# Tarea 2.2.6 - Recalculo de Tabla 11 (class-blind)
# ============================================================
from skimage.feature import local_binary_pattern
from skimage.filters import sobel
from statsmodels.stats.multitest import multipletests

def shannon_entropy(img):
    hist, _ = np.histogram(img, bins=256, range=(0,256), density=True)
    hist = hist[hist > 0]
    return float(-np.sum(hist * np.log2(hist)))

def edge_density(img):
    edges = sobel(img.astype(float) / 255.0)
    thresh = edges.mean() + edges.std()   # umbral adaptativo simple
    return float((edges > thresh).mean())

def lbp_energy(img, P=24, R=3):
    lbp = local_binary_pattern(img, P, R, method='uniform')
    hist, _ = np.histogram(lbp, bins=np.arange(0, P + 3), density=True)
    return float(np.sum(hist ** 2))

TECHNIQUE_PREFIX = {'gray_scott': 'rd', 'perlin': 'perlin', 'lorenz': 'lorenz'}

def build_path(technique, split, i, label):
    prefix = TECHNIQUE_PREFIX[technique]
    fname = f'{prefix}_{split}_{i:04d}_label{label}.png'
    return os.path.join(CLASSBLIND_BASE, technique, fname)

def compute_all_features(technique, split='train'):
    y = umap_data[split]['labels']
    feats = {'depresivo': {'entropy': [], 'contrast': [], 'edges': [], 'lbp': []},
             'no_depresivo': {'entropy': [], 'contrast': [], 'edges': [], 'lbp': []}}
    for i in range(len(y)):
        label = int(y[i])
        key = 'depresivo' if label == 1 else 'no_depresivo'
        path = build_path(technique, split, i, label)
        if not os.path.exists(path):
            continue
        img = np.array(Image.open(path).convert('L'))
        feats[key]['entropy'].append(shannon_entropy(img))
        feats[key]['contrast'].append(img.std())
        feats[key]['edges'].append(edge_density(img))
        feats[key]['lbp'].append(lbp_energy(img))
    return feats

# --- Correr para las 3 tecnicas y armar la tabla con correccion de Holm ---
all_pvalues, rows = [], []
for technique in ['gray_scott', 'perlin', 'lorenz']:
    feats = compute_all_features(technique, split='train')
    for metric in ['entropy', 'contrast', 'edges', 'lbp']:
        a = np.array(feats['depresivo'][metric])
        b = np.array(feats['no_depresivo'][metric])
        if len(a) == 0 or len(b) == 0:
            continue
        t, p = stats.ttest_ind(a, b, equal_var=False)
        d_cohen = (a.mean() - b.mean()) / np.sqrt((a.std()**2 + b.std()**2) / 2)  # effect size
        rows.append({'technique': technique, 'metric': metric,
                     'dep_mean': a.mean(), 'dep_std': a.std(),
                     'nodep_mean': b.mean(), 'nodep_std': b.std(),
                     't': t, 'p': p, 'cohen_d': d_cohen})
        all_pvalues.append(p)

# Correccion de Holm sobre las 12 comparaciones
reject, p_holm, _, _ = multipletests(all_pvalues, alpha=0.05, method='holm')
for row, p_h, rej in zip(rows, p_holm, reject):
    row['p_holm'] = p_h
    row['sig_holm'] = rej

import pandas as pd
df_table11 = pd.DataFrame(rows)
print(df_table11.to_string(index=False))


 technique   metric  dep_mean  dep_std  nodep_mean  nodep_std          t            p   cohen_d       p_holm  sig_holm
gray_scott  entropy  3.272000 1.148521    4.683233   1.409402 -16.479997 3.551412e-53 -1.097727 3.906553e-52      True
gray_scott contrast 56.430337 7.347023   59.800974   9.575157  -5.969305 3.460247e-09 -0.394960 2.076148e-08      True
gray_scott    edges  0.161117 0.029741    0.183707   0.038505  -9.916701 4.898775e-22 -0.656618 4.408897e-21      True
gray_scott      lbp  0.519305 0.151388    0.314287   0.159234  19.470200 4.525616e-69  1.319624 5.430740e-68      True
    perlin  entropy  2.789760 0.622147    2.754437   0.495055   0.899533 3.687138e-01  0.062830 7.374277e-01     False
    perlin contrast 65.483705 6.250287   63.869206   6.264594   3.786288 1.651299e-04  0.258013 8.256493e-04      True
    perlin    edges  0.174063 0.017301    0.159394   0.013511  13.504951 1.309439e-36  0.945056 1.309439e-35      True
    perlin      lbp  0.553828 0.090924    0.5588

In [17]:
# ============================================================
# Test de brillo (baseline simple) sobre las 3 técnicas CLASS-BLIND
# NOTA: esto ya NO es un "test de fuga" -- el renderer ya está verificado
# como ciego a la etiqueta. Esto es solo un baseline extra de referencia,
# igual de simple que el de la Tarea 2.1, para comparar despues contra el CNN.
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def collect_brightness(technique, split):
    y = umap_data[split]['labels']
    means = []
    labels_used = []
    for i in range(len(y)):
        label = int(y[i])
        path = build_path(technique, split, i, label)
        if not os.path.exists(path):
            continue
        img = np.array(Image.open(path).convert('L')).astype(float)
        means.append(img.mean())
        labels_used.append(label)
    return np.array(means).reshape(-1, 1), np.array(labels_used)

print(f"{'Técnica':<12} {'F1-train':<10} {'F1-test':<10}")
for technique in ['gray_scott', 'perlin', 'lorenz']:
    X_train_b, y_train_b = collect_brightness(technique, 'train')
    X_test_b,  y_test_b  = collect_brightness(technique, 'test')

    clf = LogisticRegression(class_weight='balanced').fit(X_train_b, y_train_b)
    f1_train = f1_score(y_train_b, clf.predict(X_train_b), average='macro')
    f1_test  = f1_score(y_test_b,  clf.predict(X_test_b),  average='macro')

    print(f"{technique:<12} {f1_train:<10.4f} {f1_test:<10.4f}")

Técnica      F1-train   F1-test   
gray_scott   0.6290     0.5290    
perlin       0.5362     0.4534    
lorenz       0.6072     0.4735    


## Etapa 2.2.7 — Entrenamiento del CNN (siguiente paso, aún no iniciado)


In [18]:
# ============================================================
# Etapa 2.2.7 - Bloque 1: Dataset de PyTorch para Gray-Scott class-blind
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Usando device:', device)

class GrayScottDataset(Dataset):
    def __init__(self, base_dir, split, labels, transform=None):
        self.base_dir = base_dir
        self.split = split
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        label = int(self.labels[idx])
        fname = f'rd_{self.split}_{idx:04d}_label{label}.png'
        path = os.path.join(self.base_dir, self.split, fname)
        img = Image.open(path).convert('L').convert('RGB')  # 3 canales (R=G=B) para ResNet preentrenado
        if self.transform:
            img = self.transform(img)
        return img, label

# Transforms: resize a 224x224 (tamaño esperado por ResNet), normalizacion estandar de ImageNet
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),   # augmentation leve (geometria, no afecta el argumento de fuga)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

GS_DIR = os.path.join(CLASSBLIND_BASE, 'gray_scott')  # ya no tiene subcarpetas train/val/test dentro? -- ajusta abajo si aplica

# OJO: en nuestra generacion, todos los archivos de un split quedaron en la MISMA carpeta GS_OUTPUT
# (no en subcarpetas train/val/test). Ajustamos el Dataset para reflejar eso:
class GrayScottDatasetFlat(Dataset):
    def __init__(self, base_dir, split, labels, transform=None):
        self.base_dir = base_dir  # aqui ya es la carpeta plana (gray_scott/), sin subcarpeta de split
        self.split = split
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        label = int(self.labels[idx])
        fname = f'rd_{self.split}_{idx:04d}_label{label}.png'
        path = os.path.join(self.base_dir, fname)
        img = Image.open(path).convert('L').convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

train_ds = GrayScottDatasetFlat(GS_DIR, 'train', umap_data['train']['labels'], transform=train_transform)
val_ds   = GrayScottDatasetFlat(GS_DIR, 'val',   umap_data['val']['labels'],   transform=eval_transform)
test_ds  = GrayScottDatasetFlat(GS_DIR, 'test',  umap_data['test']['labels'],  transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

# Sanity check rapido
imgs, labels = next(iter(train_loader))
print('Batch shape:', imgs.shape, '| labels ejemplo:', labels[:10])

Usando device: cuda
Batch shape: torch.Size([32, 3, 224, 224]) | labels ejemplo: tensor([0, 0, 0, 0, 0, 0, 1, 1, 0, 0])


In [19]:
# ============================================================
# Etapa 2.2.7 - Bloque 2: ResNet-18 (transfer learning, fine-tuning completo)
# ============================================================
from sklearn.metrics import f1_score
import copy

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# Pesos de clase (dataset desbalanceado 555/353 en train)
n_nodep, n_dep = (umap_data['train']['labels'] == 0).sum(), (umap_data['train']['labels'] == 1).sum()
class_weights = torch.tensor([1.0 / n_nodep, 1.0 / n_dep], dtype=torch.float32).to(device)
class_weights = class_weights / class_weights.sum() * 2

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

N_EPOCHS = 15
PATIENCE = 4
best_f1_val = 0.0
best_state = None
epochs_no_improve = 0

for epoch in range(N_EPOCHS):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

    # Validacion
    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
    f1_val = f1_score(val_true, val_preds, average='macro')

    print(f'Epoch {epoch+1}/{N_EPOCHS} - loss={running_loss/len(train_ds):.4f} - F1-val={f1_val:.4f}')

    if f1_val > best_f1_val:
        best_f1_val = f1_val
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'Early stopping en epoch {epoch+1} (mejor F1-val={best_f1_val:.4f})')
            break

model.load_state_dict(best_state)
print(f'\nMejor F1-val: {best_f1_val:.4f}')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 191MB/s]


Epoch 1/15 - loss=0.4183 - F1-val=0.7039
Epoch 2/15 - loss=0.3238 - F1-val=0.6590
Epoch 3/15 - loss=0.2741 - F1-val=0.7119
Epoch 4/15 - loss=0.2502 - F1-val=0.7173
Epoch 5/15 - loss=0.2151 - F1-val=0.6747
Epoch 6/15 - loss=0.1684 - F1-val=0.6882
Epoch 7/15 - loss=0.1217 - F1-val=0.6879
Epoch 8/15 - loss=0.0824 - F1-val=0.6372
Early stopping en epoch 8 (mejor F1-val=0.7173)

Mejor F1-val: 0.7173


In [20]:
# ============================================================
# Etapa 2.2.7 - Bloque 3: Evaluacion en test + comparacion contra baselines
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
test_preds, test_true = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_true.extend(labels.numpy())

f1_test_cnn = f1_score(test_true, test_preds, average='macro')
print(f'F1-Macro CNN (test): {f1_test_cnn:.4f}\n')
print(classification_report(test_true, test_preds, target_names=['No Depresivo', 'Depresivo']))
print('Matriz de confusion:\n', confusion_matrix(test_true, test_preds))

print('\n=== COMPARACION FINAL ===')
print(f"{'Metodo':<25} {'F1-Test':<10}")
print(f"{'BETO (texto, linea base)':<25} {'0.7586':<10}")
print(f"{'LogReg 5D-UMAP':<25} {'0.7246':<10}")
print(f"{'SVM 5D-UMAP':<25} {'0.7246':<10}")
print(f"{'RF 5D-UMAP':<25} {'0.7304':<10}")
print(f"{'MLP 5D-UMAP':<25} {'0.6933':<10}")
print(f"{'Brillo promedio (imagen)':<25} {'0.5290':<10}")
print(f"{'CNN Gray-Scott (imagen)':<25} {f1_test_cnn:<10.4f}")

# Guardar predicciones para el test de McNemar (siguiente paso)
np.save('cnn_test_preds.npy', np.array(test_preds))
np.save('cnn_test_true.npy', np.array(test_true))

F1-Macro CNN (test): 0.6685

              precision    recall  f1-score   support

No Depresivo       0.73      0.79      0.76        70
   Depresivo       0.62      0.55      0.58        44

    accuracy                           0.69       114
   macro avg       0.67      0.67      0.67       114
weighted avg       0.69      0.69      0.69       114

Matriz de confusion:
 [[55 15]
 [20 24]]

=== COMPARACION FINAL ===
Metodo                    F1-Test   
BETO (texto, linea base)  0.7586    
LogReg 5D-UMAP            0.7246    
SVM 5D-UMAP               0.7246    
RF 5D-UMAP                0.7304    
MLP 5D-UMAP               0.6933    
Brillo promedio (imagen)  0.5290    
CNN Gray-Scott (imagen)   0.6685    


In [21]:
# ============================================================
# Prueba de McNemar: CNN vs. LogReg-5D-UMAP (mismo test set, n=114)
# ============================================================
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

# Predicciones del CNN (ya guardadas)
cnn_preds = np.load('cnn_test_preds.npy')
y_test_true = np.load('cnn_test_true.npy')

# Predicciones del LogReg-5D-UMAP sobre el MISMO test set
# (reusa 'classifiers' y X_test, y_test de la Etapa 1 si siguen en memoria;
#  si no, recrea el LogReg rapido)
logreg_5d = classifiers['LogReg']  # ya entrenado en la Etapa 1
logreg_preds = logreg_5d.predict(X_test)

assert len(cnn_preds) == len(logreg_preds) == len(y_test_true), "Los arrays deben tener el mismo largo y orden"

# Tabla de contingencia 2x2: correcto/incorrecto de cada modelo
cnn_correct = (cnn_preds == y_test_true)
logreg_correct = (logreg_preds == y_test_true)

n_both_correct   = np.sum(cnn_correct & logreg_correct)
n_only_cnn       = np.sum(cnn_correct & ~logreg_correct)
n_only_logreg    = np.sum(~cnn_correct & logreg_correct)
n_both_wrong     = np.sum(~cnn_correct & ~logreg_correct)

table = [[n_both_correct, n_only_cnn],
         [n_only_logreg, n_both_wrong]]

print('Tabla de contingencia:')
print(f'  Ambos correctos: {n_both_correct}')
print(f'  Solo CNN correcto: {n_only_cnn}')
print(f'  Solo LogReg-5D correcto: {n_only_logreg}')
print(f'  Ambos incorrectos: {n_both_wrong}')

result = mcnemar(table, exact=True)
print(f'\nMcNemar: statistic={result.statistic}, p-value={result.pvalue:.4f}')
if result.pvalue < 0.05:
    print('=> Diferencia estadisticamente significativa entre CNN y LogReg-5D-UMAP')
else:
    print('=> NO hay diferencia estadisticamente significativa (los dos metodos son equivalentes)')

Tabla de contingencia:
  Ambos correctos: 70
  Solo CNN correcto: 9
  Solo LogReg-5D correcto: 14
  Ambos incorrectos: 21

McNemar: statistic=9.0, p-value=0.4049
=> NO hay diferencia estadisticamente significativa (los dos metodos son equivalentes)


In [22]:
# ============================================================
# K-Fold - Bloque 1: combinar los 3 splits en un solo conjunto
# ============================================================
from sklearn.model_selection import StratifiedKFold, train_test_split

def combine_splits():
    all_umap, all_labels, all_paths = [], [], []
    for split in ['train', 'val', 'test']:
        X = umap_data[split]['params']
        y = umap_data[split]['labels']
        for i in range(len(y)):
            label = int(y[i])
            fname = f'rd_{split}_{i:04d}_label{label}.png'
            all_umap.append(X[i])
            all_labels.append(label)
            all_paths.append(os.path.join(GS_DIR, fname))
    return np.array(all_umap), np.array(all_labels), np.array(all_paths)

X_all, y_all, paths_all = combine_splits()
print('Total combinado:', len(y_all), '| balance:', np.bincount(y_all))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Total combinado: 1136 | balance: [695 441]


In [23]:
# ============================================================
# K-Fold - Bloque 2: baselines 5D-UMAP
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
from sklearn.base import clone

classifiers_kfold = {
    'LogReg': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'SVM':    SVC(kernel='rbf', random_state=42, class_weight='balanced'),
    'RF':     RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'),
    'MLP':    MLPClassifier(hidden_layer_sizes=(32,), max_iter=2000, random_state=42),
}

kfold_results_umap = {name: [] for name in classifiers_kfold}

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
    for name, clf_template in classifiers_kfold.items():
        clf = clone(clf_template)
        clf.fit(X_all[train_idx], y_all[train_idx])
        f1 = f1_score(y_all[test_idx], clf.predict(X_all[test_idx]), average='macro')
        kfold_results_umap[name].append(f1)
    print(f'Fold {fold_idx+1}/5 completado')

print('\n=== Resultados 5-fold (5D UMAP) ===')
for name, scores in kfold_results_umap.items():
    scores = np.array(scores)
    print(f'{name:<10} F1-Macro: {scores.mean():.4f} +- {scores.std():.4f}  (folds: {np.round(scores,4)})')

Fold 1/5 completado
Fold 2/5 completado
Fold 3/5 completado
Fold 4/5 completado
Fold 5/5 completado

=== Resultados 5-fold (5D UMAP) ===
LogReg     F1-Macro: 0.8274 +- 0.0230  (folds: [0.8153 0.8457 0.8237 0.8589 0.7935])
SVM        F1-Macro: 0.8215 +- 0.0174  (folds: [0.8111 0.8414 0.8237 0.8373 0.794 ])
RF         F1-Macro: 0.8241 +- 0.0149  (folds: [0.8334 0.8337 0.8123 0.8401 0.8009])
MLP        F1-Macro: 0.8312 +- 0.0248  (folds: [0.8263 0.8663 0.8101 0.8524 0.8009])


In [24]:
# ============================================================
# K-Fold - Bloque 3: CNN Gray-Scott
# ============================================================
import copy

class GrayScottDatasetFromList(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths, self.labels, self.transform = paths, labels, transform
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('L').convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])

kfold_results_cnn = []

for fold_idx, (trainval_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
    y_trainval = y_all[trainval_idx]
    train_idx, val_idx = train_test_split(trainval_idx, test_size=0.1, stratify=y_trainval, random_state=42)

    train_ds_f = GrayScottDatasetFromList(paths_all[train_idx], y_all[train_idx], transform=train_transform)
    val_ds_f   = GrayScottDatasetFromList(paths_all[val_idx],   y_all[val_idx],   transform=eval_transform)
    test_ds_f  = GrayScottDatasetFromList(paths_all[test_idx],  y_all[test_idx],  transform=eval_transform)

    train_loader_f = DataLoader(train_ds_f, batch_size=32, shuffle=True,  num_workers=2)
    val_loader_f   = DataLoader(val_ds_f,   batch_size=32, shuffle=False, num_workers=2)
    test_loader_f  = DataLoader(test_ds_f,  batch_size=32, shuffle=False, num_workers=2)

    model_f = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model_f.fc = nn.Linear(model_f.fc.in_features, 2)
    model_f = model_f.to(device)

    n0, n1 = (y_all[train_idx]==0).sum(), (y_all[train_idx]==1).sum()
    cw = torch.tensor([1.0/n0, 1.0/n1], dtype=torch.float32).to(device)
    cw = cw / cw.sum() * 2
    criterion_f = nn.CrossEntropyLoss(weight=cw)
    optimizer_f = torch.optim.Adam(model_f.parameters(), lr=1e-4)

    best_f1_val_f, best_state_f, no_improve = 0.0, None, 0
    for epoch in range(15):
        model_f.train()
        for imgs, labels in train_loader_f:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer_f.zero_grad()
            loss = criterion_f(model_f(imgs), labels)
            loss.backward(); optimizer_f.step()

        model_f.eval()
        vp, vt = [], []
        with torch.no_grad():
            for imgs, labels in val_loader_f:
                preds = model_f(imgs.to(device)).argmax(dim=1).cpu().numpy()
                vp.extend(preds); vt.extend(labels.numpy())
        f1v = f1_score(vt, vp, average='macro')

        if f1v > best_f1_val_f:
            best_f1_val_f, best_state_f, no_improve = f1v, copy.deepcopy(model_f.state_dict()), 0
        else:
            no_improve += 1
            if no_improve >= 4:
                break

    model_f.load_state_dict(best_state_f)
    model_f.eval()
    tp, tt = [], []
    with torch.no_grad():
        for imgs, labels in test_loader_f:
            preds = model_f(imgs.to(device)).argmax(dim=1).cpu().numpy()
            tp.extend(preds); tt.extend(labels.numpy())
    f1_test_f = f1_score(tt, tp, average='macro')
    kfold_results_cnn.append(f1_test_f)
    print(f'Fold {fold_idx+1}/5 - CNN F1-test={f1_test_f:.4f} (mejor F1-val={best_f1_val_f:.4f})')

kfold_results_cnn = np.array(kfold_results_cnn)
print(f'\nCNN 5-fold F1-Macro: {kfold_results_cnn.mean():.4f} +- {kfold_results_cnn.std():.4f}')
print('Folds individuales:', np.round(kfold_results_cnn, 4))

Fold 1/5 - CNN F1-test=0.7967 (mejor F1-val=0.8513)
Fold 2/5 - CNN F1-test=0.8358 (mejor F1-val=0.8178)
Fold 3/5 - CNN F1-test=0.8256 (mejor F1-val=0.8037)
Fold 4/5 - CNN F1-test=0.8213 (mejor F1-val=0.8299)
Fold 5/5 - CNN F1-test=0.7454 (mejor F1-val=0.8607)

CNN 5-fold F1-Macro: 0.8050 +- 0.0325
Folds individuales: [0.7967 0.8358 0.8256 0.8213 0.7454]


In [25]:
# ============================================================
# Bloque A - Setup: holdout final + funciones reutilizables
# ============================================================
import joblib, numpy as np, os, time, copy
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score
import umap

# --- Cargar embeddings 768D ---
beto = joblib.load(os.path.join(DRIVE_ROOT, 'data/shared/Exp_4.1_BETO_HPO_embeddings.pkl'))
X_768_all = np.concatenate([beto['X_train'], beto['X_val'], beto['X_test']], axis=0)
y_all_768 = np.concatenate([beto['y_train'], beto['y_val'], beto['y_test']], axis=0)
print('Total:', X_768_all.shape, '| balance:', np.bincount(y_all_768))

# --- Separar HOLDOUT FINAL (10%) -- nunca se toca hasta el final ---
dev_idx, holdout_idx = train_test_split(
    np.arange(len(y_all_768)), test_size=0.10, stratify=y_all_768, random_state=42
)
np.save('dev_idx.npy', dev_idx)
np.save('holdout_idx.npy', holdout_idx)
print(f'Dev: {len(dev_idx)} | Holdout: {len(holdout_idx)} (guardados en dev_idx.npy / holdout_idx.npy)')

X_dev, y_dev = X_768_all[dev_idx], y_all_768[dev_idx]
X_holdout, y_holdout = X_768_all[holdout_idx], y_all_768[holdout_idx]

# --- Funciones reutilizables ---
def fit_umap_scaler(X_fit, random_state=42):
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', n_components=5, random_state=random_state)
    coords_fit = reducer.fit_transform(X_fit)
    scaler = MinMaxScaler().fit(coords_fit)
    return reducer, scaler, scaler.transform(coords_fit)

def transform_umap_scaler(reducer, scaler, X):
    return scaler.transform(reducer.transform(X))

def generate_gray_scott_batch(coords_5d, labels, tag, output_dir):
    """tag: string unico para nombrar archivos (ej. 'fold0_train', 'final_holdout')."""
    os.makedirs(output_dir, exist_ok=True)
    paths = []
    for i in range(len(labels)):
        label = int(labels[i])
        fname = f'{tag}_{i:04d}_label{label}.png'
        fpath = os.path.join(output_dir, fname)
        paths.append(fpath)
        if os.path.exists(fpath):
            continue
        _, _, _ = generate_and_save_grayscale(i, coords_5d[i], label, tag, output_dir)
    return paths

print('Bloque A listo.')

Total: (1136, 768) | balance: [695 441]
Dev: 1022 | Holdout: 114 (guardados en dev_idx.npy / holdout_idx.npy)
Bloque A listo.


In [26]:
# ============================================================
# Bloque B (VERSION RESUMIBLE) - reemplaza el bloque B anterior por este
# ============================================================
import json

RESULTS_FILE = 'nested_cv_results.json'
NESTED_BASE = os.path.join(CLASSBLIND_BASE, 'gray_scott_nested_cv')

# Cargar resultados previos si existen (para reanudar)
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        nested_results = json.load(f)
    print(f'Reanudando -- folds ya completados: {list(nested_results.keys())}')
else:
    nested_results = {}

def save_results():
    with open(RESULTS_FILE, 'w') as f:
        json.dump(nested_results, f, indent=2)

skf_dev = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf_dev.split(X_dev, y_dev))

for fold_idx, (tr_idx, te_idx) in enumerate(folds):
    fold_key = f'fold{fold_idx}'
    if fold_key in nested_results:
        print(f'Fold {fold_idx+1}/5 ya completado (F1_cnn={nested_results[fold_key]["cnn"]:.4f}) -- saltando')
        continue

    print(f'\n{"="*50}\nFOLD {fold_idx+1}/5\n{"="*50}')
    t_fold_start = time.time()

    reducer_f, scaler_f, coords_train_f = fit_umap_scaler(X_dev[tr_idx], random_state=42)
    coords_test_f = transform_umap_scaler(reducer_f, scaler_f, X_dev[te_idx])

    fold_umap_results = {}
    classifiers_f = {
        'LogReg': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        'SVM':    SVC(kernel='rbf', random_state=42, class_weight='balanced'),
        'RF':     RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'),
        'MLP':    MLPClassifier(hidden_layer_sizes=(32,), max_iter=2000, random_state=42),
    }
    for name, clf_t in classifiers_f.items():
        clf = clone(clf_t)
        clf.fit(coords_train_f, y_dev[tr_idx])
        f1 = f1_score(y_dev[te_idx], clf.predict(coords_test_f), average='macro')
        fold_umap_results[name] = f1
        print(f'  {name}: F1={f1:.4f}')

    fold_dir = os.path.join(NESTED_BASE, f'fold{fold_idx}')
    paths_train_f = generate_gray_scott_batch(coords_train_f, y_dev[tr_idx], f'f{fold_idx}train', fold_dir)
    paths_test_f  = generate_gray_scott_batch(coords_test_f,  y_dev[te_idx],  f'f{fold_idx}test',  fold_dir)
    print(f'  Imagenes listas ({time.time()-t_fold_start:.0f}s)')

    y_tr_f = y_dev[tr_idx]
    tr_sub_idx, val_sub_idx = train_test_split(np.arange(len(tr_idx)), test_size=0.1, stratify=y_tr_f, random_state=42)

    train_ds_f = GrayScottDatasetFromList(np.array(paths_train_f)[tr_sub_idx], y_tr_f[tr_sub_idx], transform=train_transform)
    val_ds_f   = GrayScottDatasetFromList(np.array(paths_train_f)[val_sub_idx], y_tr_f[val_sub_idx], transform=eval_transform)
    test_ds_f  = GrayScottDatasetFromList(np.array(paths_test_f), y_dev[te_idx], transform=eval_transform)

    train_loader_f = DataLoader(train_ds_f, batch_size=32, shuffle=True, num_workers=2)
    val_loader_f   = DataLoader(val_ds_f,   batch_size=32, shuffle=False, num_workers=2)
    test_loader_f  = DataLoader(test_ds_f,  batch_size=32, shuffle=False, num_workers=2)

    model_f = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model_f.fc = nn.Linear(model_f.fc.in_features, 2)
    model_f = model_f.to(device)

    n0, n1 = (y_tr_f[tr_sub_idx]==0).sum(), (y_tr_f[tr_sub_idx]==1).sum()
    cw = torch.tensor([1.0/n0, 1.0/n1], dtype=torch.float32).to(device)
    cw = cw / cw.sum() * 2
    criterion_f = nn.CrossEntropyLoss(weight=cw)
    optimizer_f = torch.optim.Adam(model_f.parameters(), lr=3e-5)

    best_f1v, best_state, no_improve = 0.0, None, 0
    for epoch in range(15):
        model_f.train()
        for imgs, labels in train_loader_f:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer_f.zero_grad()
            loss = criterion_f(model_f(imgs), labels)
            loss.backward(); optimizer_f.step()
        model_f.eval()
        vp, vt = [], []
        with torch.no_grad():
            for imgs, labels in val_loader_f:
                preds = model_f(imgs.to(device)).argmax(dim=1).cpu().numpy()
                vp.extend(preds); vt.extend(labels.numpy())
        f1v = f1_score(vt, vp, average='macro')
        if f1v > best_f1v:
            best_f1v, best_state, no_improve = f1v, copy.deepcopy(model_f.state_dict()), 0
        else:
            no_improve += 1
            if no_improve >= 4:
                break

    model_f.load_state_dict(best_state)
    model_f.eval()
    tp, tt = [], []
    with torch.no_grad():
        for imgs, labels in test_loader_f:
            preds = model_f(imgs.to(device)).argmax(dim=1).cpu().numpy()
            tp.extend(preds); tt.extend(labels.numpy())
    f1_test_f = f1_score(tt, tp, average='macro')

    # GUARDAR INMEDIATAMENTE -- esto es lo que nos salva si se corta la sesion
    nested_results[fold_key] = {**fold_umap_results, 'cnn': f1_test_f}
    save_results()
    print(f'  Fold {fold_idx+1} completo y GUARDADO: CNN F1={f1_test_f:.4f} ({(time.time()-t_fold_start)/60:.1f} min)')

# Resumen final (solo si los 5 folds estan completos)
if len(nested_results) == 5:
    print('\n=== RESULTADOS FINALES (5/5 folds) ===')
    for name in ['LogReg','SVM','RF','MLP','cnn']:
        scores = np.array([nested_results[f'fold{i}'][name] for i in range(5)])
        print(f'{name:<10} F1: {scores.mean():.4f} +- {scores.std():.4f}')
else:
    print(f'\nFaltan folds: {5 - len(nested_results)}/5 por completar. Vuelve a correr esta celda para continuar.')


FOLD 1/5


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  LogReg: F1=0.8498
  SVM: F1=0.8498
  RF: F1=0.7950
  MLP: F1=0.8673


/tmp/ipykernel_58/1363527457.py:52: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(gray, mode='L').save(fpath)


OSError: [Errno 30] Read-only file system: '/kaggle/input/datasets/palaciosroblesdiego/data-files/TT/images_grayscale_classblind/gray_scott_nested_cv/fold0/rd_f0train_0000_label0.png'

In [ ]:
# ============================================================
# Bloque C - Modelo FINAL: UMAP+CNN sobre todo 'dev', evaluado 1 vez en holdout
# ============================================================
reducer_final, scaler_final, coords_dev_final = fit_umap_scaler(X_dev, random_state=42)
coords_holdout_final = transform_umap_scaler(reducer_final, scaler_final, X_holdout)

FINAL_DIR = os.path.join(CLASSBLIND_BASE, 'gray_scott_final_model')
paths_dev_final = generate_gray_scott_batch(coords_dev_final, y_dev, 'dev', FINAL_DIR)
paths_holdout_final = generate_gray_scott_batch(coords_holdout_final, y_holdout, 'holdout', FINAL_DIR)

tr_idx_f, val_idx_f = train_test_split(np.arange(len(y_dev)), test_size=0.1, stratify=y_dev, random_state=42)

train_ds_final = GrayScottDatasetFromList(np.array(paths_dev_final)[tr_idx_f], y_dev[tr_idx_f], transform=train_transform)
val_ds_final   = GrayScottDatasetFromList(np.array(paths_dev_final)[val_idx_f], y_dev[val_idx_f], transform=eval_transform)
holdout_ds     = GrayScottDatasetFromList(np.array(paths_holdout_final), y_holdout, transform=eval_transform)

train_loader_final = DataLoader(train_ds_final, batch_size=32, shuffle=True, num_workers=2)
val_loader_final   = DataLoader(val_ds_final,   batch_size=32, shuffle=False, num_workers=2)
holdout_loader     = DataLoader(holdout_ds,     batch_size=32, shuffle=False, num_workers=2)

model_final = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model_final.fc = nn.Linear(model_final.fc.in_features, 2)
model_final = model_final.to(device)

n0, n1 = (y_dev[tr_idx_f]==0).sum(), (y_dev[tr_idx_f]==1).sum()
cw = torch.tensor([1.0/n0, 1.0/n1], dtype=torch.float32).to(device)
cw = cw / cw.sum() * 2
criterion = nn.CrossEntropyLoss(weight=cw)
optimizer = torch.optim.Adam(model_final.parameters(), lr=3e-5)

best_f1v, best_state, no_improve = 0.0, None, 0
for epoch in range(15):
    model_final.train()
    for imgs, labels in train_loader_final:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_final(imgs), labels)
        loss.backward(); optimizer.step()
    model_final.eval()
    vp, vt = [], []
    with torch.no_grad():
        for imgs, labels in val_loader_final:
            preds = model_final(imgs.to(device)).argmax(dim=1).cpu().numpy()
            vp.extend(preds); vt.extend(labels.numpy())
    f1v = f1_score(vt, vp, average='macro')
    print(f'Epoch {epoch+1}: F1-val={f1v:.4f}')
    if f1v > best_f1v:
        best_f1v, best_state, no_improve = f1v, copy.deepcopy(model_final.state_dict()), 0
    else:
        no_improve += 1
        if no_improve >= 4:
            break

model_final.load_state_dict(best_state)
model_final.eval()
hp, ht = [], []
with torch.no_grad():
    for imgs, labels in holdout_loader:
        preds = model_final(imgs.to(device)).argmax(dim=1).cpu().numpy()
        hp.extend(preds); ht.extend(labels.numpy())

f1_holdout = f1_score(ht, hp, average='macro')
print(f'\n*** F1-Macro FINAL sobre HOLDOUT (nunca antes visto) = {f1_holdout:.4f} ***')